<a href="https://colab.research.google.com/github/alessarana90-hue/ML/blob/main/magMachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import important library

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
!pip install tensorflow

In [ ]:
# Step 1: Import important libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Dropout
from tensorflow.keras.metrics import MeanSquaredError, MeanAbsoluteError
from tensorflow.keras.layers import Dense, Dropout, LSTM, Bidirectional, Input, BatchNormalization
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
import math
# Import necessary libraries for each model
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error

# Upload dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/The Pacific Ring of Fire.csv")

In [ ]:
df .head(5)

,id,time,latitude,longitude,depth,magnitude,place,tsunami,alert,felt,...,status,sig,net,nst,dmin,gap,magType,type,title,location
0,us6000pfs7,12/26/2024 21:02,30.4847,141.8463,10.000,5.7,"Izu Islands, Japan region",0,green,NaN,...,reviewed,500,us,103.0,3.142,92.0,mww,earthquake,"M 5.7 - Izu Islands, Japan region","Izu Islands, Japan region"
1,us7000nzg7,12/17/2024 4:09,31.0904,130.2373,165.486,5.2,"20 km SSW of Makurazaki, Japan",0,NaN,1.0,...,reviewed,416,us,88.0,0.456,82.0,mb,earthquake,"M 5.2 - 20 km SSW of Makurazaki, Japan","20 km SSW of Makurazaki, Japan"
2,us7000ny8l,12/12/2024 19:55,30.5632,141.9457,10.000,5.1,"Izu Islands, Japan region",0,NaN,NaN,...,reviewed,400,us,87.0,3.125,119.0,mb,earthquake,"M 5.1 - Izu Islands, Japan region","Izu Islands, Japan region"
3,us7000nvxl,12/4/2024 16:53,24.8543,128.0555,10.000,5.0,"146 km SSE of Itoman, Japan",0,NaN,NaN,...,reviewed,385,us,120.0,1.983,103.0,mb,earthquake,"M 5.0 - 146 km SSE of Itoman, Japan","146 km SSE of Itoman, Japan"
4,us7000nv3a,11/30/2024 8:46,25.3021,126.0218,49.015,5.7,"91 km NE of Hirara, Japan",0,green,3.0,...,reviewed,501,us,257.0,1.866,62.0,mww,earthquake,"M 5.7 - 91 km NE of Hirara, Japan","91 km NE of Hirara, Japan"


In [ ]:
df.shape

(27759, 23)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27759 entries, 0 to 27758
Data columns (total 23 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         27759 non-null  object 
 1   time       27759 non-null  object 
 2   latitude   27759 non-null  float64
 3   longitude  27759 non-null  float64
 4   depth      27759 non-null  float64
 5   magnitude  27759 non-null  float64
 6   place      27759 non-null  object 
 7   tsunami    27759 non-null  int64  
 8   alert      1480 non-null   object 
 9   felt       3221 non-null   float64
 10  cdi        3221 non-null   float64
 11  mmi        5460 non-null   float64
 12  country    27759 non-null  object 
 13  status     27759 non-null  object 
 14  sig        27759 non-null  int64  
 15  net        27759 non-null  object 
 16  nst        7991 non-null   float64
 17  dmin       4468 non-null   float64
 18  gap        10205 non-null  float64
 19  magType    27759 non-null  object 
 20  type  

In [ ]:
print("Missing values in each feature:")
print(df.isnull().sum())

Missing values in each feature:
id               0
time             0
latitude         0
longitude        0
depth            0
magnitude        0
place            0
tsunami          0
alert        26279
felt         24538
cdi          24538
mmi          22299
country          0
status           0
sig              0
net              0
nst          19768
dmin         23291
gap          17554
magType          0
type             0
title            0
location         0
dtype: int64


# Preprocessing

In [ ]:
# Handle remaining missing values
# Fill numeric columns with the mean
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

In [ ]:
# Fill categorical columns with the mode (most frequent value)
categorical_cols = df.select_dtypes(include=[object]).columns
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Detect and manage outliers using IQR
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df

# Apply outlier removal to numerical columns
for col in numeric_cols:
    df = remove_outliers(df, col)

<ipython-input-242-969f4d23c9fb>:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
df.shape

(9780, 23)

In [ ]:
# Handle categorical data (assuming 'description' is a column with string values)
#if 'description' in df.columns:
#    df.drop(['description'], axis=1, inplace=True)

In [ ]:
#df.drop(['continent'], axis=1, inplace=True)
df.drop(['latitude'], axis=1, inplace=True)
df.drop(['longitude'], axis=1, inplace=True)
df.drop(['depth'], axis=1, inplace=True)
df.drop(['alert'], axis=1, inplace=True)
df.drop(['mmi'], axis=1, inplace=True)
df.drop(['country'], axis=1, inplace=True)
df.drop(['status'], axis=1, inplace=True)
df.drop(['sig'], axis=1, inplace=True)
df.drop(['net'], axis=1, inplace=True)
df.drop(['nst'], axis=1, inplace=True)
df.drop(['dmin'], axis=1, inplace=True)
df.drop(['gap'], axis=1, inplace=True)
df.drop(['magType'], axis=1, inplace=True)
df.drop(['type'], axis=1, inplace=True)
df.drop(['title'], axis=1, inplace=True)
df.drop(['location'], axis=1, inplace=True)
df.drop(['id'], axis=1, inplace=True)

In [ ]:
df.head(5)

,time,magnitude,place,tsunami,felt,cdi
3506,5/2/2000 5:25,5.0,"133 km ENE of Miyako, Japan",0,33.967401,3.782614
3507,5/1/2000 21:05,5.0,"132 km ENE of Miyako, Japan",0,33.967401,3.782614
3508,5/1/2000 20:45,5.2,"131 km ENE of Miyako, Japan",0,33.967401,3.782614
3509,4/30/2000 12:39,5.4,"176 km ENE of Miyako, Japan",0,33.967401,3.782614
3510,4/26/2000 12:55,5.6,"125 km ENE of Miyako, Japan",0,33.967401,3.782614


In [ ]:
# Ensure all features are numeric
for column in df.columns:
    if df[column].dtype == 'object':
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])

In [ ]:
df.head(5)

,time,magnitude,place,tsunami,felt,cdi
3506,6103,5.0,1352,0,33.967401,3.782614
3507,5827,5.0,1318,0,33.967401,3.782614
3508,5826,5.2,1283,0,33.967401,3.782614
3509,5661,5.4,2727,0,33.967401,3.782614
3510,5546,5.6,1049,0,33.967401,3.782614


In [ ]:
# Step 3: Do preprocessing
# Extract features and target
X = df.drop(columns=['magnitude'])
y = df['magnitude']

In [ ]:
X.shape

(9780, 5)

In [ ]:
X.head(5)

,time,place,tsunami,felt,cdi
3506,6103,1352,0,33.967401,3.782614
3507,5827,1318,0,33.967401,3.782614
3508,5826,1283,0,33.967401,3.782614
3509,5661,2727,0,33.967401,3.782614
3510,5546,1049,0,33.967401,3.782614


In [ ]:
# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
df.shape

(9780, 6)

# Spliting Dataset

In [ ]:
# Step 1: Split data into 80% training+validation and 20% testing
X_text_train_val, X_text_test, y_train_val, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Step 2: Split the 80% training+validation set into 70% training and 10% validation
X_text_train, X_text_val, y_train, y_val = train_test_split(X_text_train_val, y_train_val, test_size=0.125, random_state=42)

# Check the shapes to ensure the correct split
print(f"Training set size: {X_text_train.shape[0]} samples")
print(f"Validation set size: {X_text_val.shape[0]} samples")
print(f"Testing set size: {X_text_test.shape[0]} samples")

Training set size: 6846 samples
Validation set size: 978 samples
Testing set size: 1956 samples


# DL Models

In [ ]:
# Helper function to train and evaluate models
def evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test):
    # Fit the model
    model.fit(X_train, y_train)

    # Evaluate on validation data
    y_val_pred = model.predict(X_val)
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_rmse = math.sqrt(val_mse)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    # Evaluate on test data
    y_test_pred = model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_rmse = math.sqrt(test_mse)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"Validation MSE: {val_mse}, MAE: {val_mae}, RMSE: {val_rmse}, R²: {val_r2}")
    print(f"Test MSE: {test_mse}, MAE: {test_mae}, RMSE: {test_rmse}, R²: {test_r2}")
    print("="*60)

    return model

 1. Support Vector Machine (SVM) with linear

In [ ]:
print("Support Vector Machine:")
svm = SVR(kernel='linear')  # Use 'linear' kernel for regression tasks
evaluate_model(svm, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Support Vector Machine:
Validation MSE: 0.07141632925701358, MAE: 0.1992644010762021, RMSE: 0.26723833792518165, R²: -0.07445707442827487
Test MSE: 0.072556998936259, MAE: 0.20288883328327087, RMSE: 0.26936406392883777, R²: -0.06298302938670819


SVR(kernel='linear')

 2. Support Vector Machine (SVM) with rbf

In [ ]:
print("Support Vector Machine:")
svm = SVR(kernel='rbf')  # Use 'linear' kernel for regression tasks
evaluate_model(svm, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Support Vector Machine:
Validation MSE: 0.0712989024111691, MAE: 0.19930477196093266, RMSE: 0.26701854319722645, R²: -0.07269039016211298
Test MSE: 0.07240897934814532, MAE: 0.20295666154469252, RMSE: 0.26908916616643136, R²: -0.0608144955100538


SVR()

3. Support Vector Machine (SVM) with poly

In [ ]:
print("Support Vector Machine:")
svm = SVR(kernel='poly')  # Use 'linear' kernel for regression tasks
evaluate_model(svm, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

Support Vector Machine:


4. Random Forest Regressor

In [ ]:

print("Random Forest Regressor:")
rf = RandomForestRegressor(n_estimators=100, random_state=42)
evaluate_model(rf, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

In [ ]:
y_test.to_csv('y_test.csv', index=False)

In [ ]:
# Print real vs predicted values for test set
y_test_pred = rf.predict(X_text_test)

print("\nTest Set - Real vs Predicted:")
for real, predicted in zip(y_test, y_test_pred):
    print(f"Real: {real}, Predicted: {predicted}")

5. Decision Tree Regressor

In [ ]:

print("Decision Tree Regressor:")
dt = DecisionTreeRegressor(random_state=42)
evaluate_model(dt, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

6. K-Nearest Neighbors (KNN) Regressor

In [ ]:

print("K-Nearest Neighbors Regressor:")
knn = KNeighborsRegressor(n_neighbors=5)  # You can tune 'n_neighbors' for better results
evaluate_model(knn, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)

7. Logistic Regression


In [ ]:
print("Logistic Regression:")
lr = LinearRegression()  # Increased iterations for convergence
evaluate_model(lr, X_text_train, X_text_val, X_text_test, y_train, y_val, y_test)